In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
#change this to your directory
base_dir = ""

daymet_mean_forcing_dir = f"{base_dir}/basin_mean_forcing/daymet"
basin_metadata_dir = f"{base_dir}/basin_metadata"
usgs_streamflow_dir = f"{base_dir}/usgs_streamflow"


In [3]:
#clean each individual directory and join

#clean daymet mean forcing - Alex
#input:  daymet_dir = path to basin_mean_forcing/daymet/
#        hucs = region folders to read, e.g. ["17"]
#output: DataFrame, one row per gauge per day
#        gauge_id (text, from the filename), date (one datetime column), weather columns
#note:   each file has 3 header lines before the column names
def clean_daymet(daymet_dir, hucs=["17"]):
    #Array for accumulating cleaned data
    daymet_data = np.empty((0, 8))

    #Column names
    daymet_data_column_names = ["gauge_id", "date", "dayl(s)", "prcp(mm/day)", "srad(W/m2)", "tmax(C)", "tmin(C)", "vp(Pa)"]

    #Loop through each huc
    for huc in hucs:
   
        #Enter the directory for each huc "{daymet_mean_forcing_dir}/{huc}"
        huc_dir = f'{daymet_dir}/{huc}'
        gauge_file_names = [file for file in os.listdir(huc_dir) if os.path.isfile(os.path.join(huc_dir, file))]
        gauge_file_paths = [
            os.path.abspath(os.path.join(huc_dir, file))
            for file in gauge_file_names
        ]
        #print(gauge_file_paths)
        
        #Loop through each file
        for i in range(len(gauge_file_paths)):
            #Extract gauge ID from filename (split the filename with _ as delimiter and take the first element) and add to dictionary as key
            gauge_id = gauge_file_names[i].split("_")[0]
            #print(f'gauge_id={gauge_id}')

            #Do a readlines operation on the file to collect raw data
            f = open(gauge_file_paths[i], "r", encoding="utf-8")
            gauge_data = f.readlines()
            f.close()

            #Collect everything except for the first four lines as day-to-day data
            gauge_data = gauge_data[4:]

            #Strip new lines
            gauge_data = [gauge_data_point.rstrip("\n") for gauge_data_point in gauge_data]

            #Split data points with \t as delimiter
            gauge_data = [gauge_data_point.split("\t") for gauge_data_point in gauge_data]

            #Remove hour measurement from date since it's always 12
            for data_point in gauge_data:
                data_point[0] = data_point[0][:-3]
            
            gauge_data = np.array(gauge_data)

            #Remove SWE (Snow-Water Equivalent) column because it is always 0
            gauge_data = np.delete(gauge_data, -4, axis=1)
            
            #Prepend gauge_id to gauge_data
            gauge_data = np.insert(gauge_data, 0, gauge_id, axis=1)

            #Add to accumulator
            daymet_data = np.vstack([daymet_data, gauge_data])
            
            

    print(f'daymet_data.shape={daymet_data.shape}') 

    #Convert to pandas dataframe to conform to integration standard
    daymet_data_df = pd.DataFrame(daymet_data, columns=daymet_data_column_names)

    #The numpy array is all text; make the six weather columns numeric
    weather_cols = daymet_data_column_names[2:]
    daymet_data_df[weather_cols] = daymet_data_df[weather_cols].astype(float)

    #Process dates to datetime Pandas datatype
    daymet_data_df['date'] = pd.to_datetime(daymet_data_df['date'], format="%Y %m %d")  

    #Return clean dataframe
    return daymet_data_df


In [4]:
#clean basin_metadata - JJ
#input:  metadata_dir = path to basin_metadata/
#        hucs = region codes to keep, e.g. ("17",) or ("17", "18")
#output: DataFrame, one row per gauge, 13 columns
#steps:
#1. gauge_information.txt -> huc, gauge_id, name, lat, lon, drainage area; keep only hucs
#2. basin_physical_characteristics.txt -> area, elevation, slope, forest fraction
#3. basin_annual_hydrometeorology_characteristics_daymet.txt -> long-term precip, PET, temp
#4. rename ids to gauge_id (kept as text for leading zeros) and merge all 3 on gauge_id
def clean_basin_metadata(metadata_dir, hucs=("17",)):
    #tab-separated, and gauge names contain spaces
    gauges = pd.read_csv(
        f"{metadata_dir}/gauge_information.txt",
        sep="\t",
        skiprows=1,
        header=None,
        names=["huc", "gauge_id", "gauge_name", "lat", "lon", "drainage_area_km2"],
        dtype={"huc": str, "gauge_id": str},
    )
    gauges["gauge_name"] = gauges["gauge_name"].str.strip()
    gauges = gauges[gauges["huc"].isin(hucs)]

    physical = pd.read_csv(
        f"{metadata_dir}/basin_physical_characteristics.txt",
        sep=r"\s+",
        dtype={"BASIN_ID": str},
    )
    physical = physical.drop(columns="BASIN_HUC").rename(columns={
        "BASIN_ID": "gauge_id",
        "Size(km2)": "area_km2",
        "Elevation(m)": "elevation_m",
        "Slope(m_km-1)": "slope_m_per_km",
        "Frac_Forest(percent)": "frac_forest",
    })

    #long-term climate averages; drop Runoff since it comes from streamflow (our label)
    climate = pd.read_csv(
        f"{metadata_dir}/basin_annual_hydrometeorology_characteristics_daymet.txt",
        sep=r"\s+",
        dtype={"BASIN_ID": str},
    )
    climate = climate.drop(columns=["BASIN_HUC", "Runoff(mm_day-1)"]).rename(columns={
        "BASIN_ID": "gauge_id",
        "Precip(mm_day-1)": "mean_precip_mm_day",
        "PET(mm_day-1)": "mean_pet_mm_day",
        "Temp(oC)": "mean_temp_c",
    })

    return gauges.merge(physical, on="gauge_id").merge(climate, on="gauge_id")

In [5]:
#clean usgs streamflow - Franzi
#input:streamflow_dir = path to usgs_streamflow/
#        hucs = region folders to read, e.g. ("17",)
#output: DataFrame, one row per gauge per day
#        gauge_id (text), date (one datetime column), flow
#note: missing flow is -999/-999.0 (flag M) and must be removed
#Franzi note:flow is measured in cubic feet/second so we might want/need to convert units later 
def clean_usgs_streamflow(streamflow_dir, hucs=("17",)):
    total_df = []
    
    #Loop through region by huc code
    for huc in hucs:
        huc_path = os.path.join(streamflow_dir, huc)
        
        for filesname in os.listdir(huc_path):
            if not filesname.endswith(".txt"):
                continue
            
            files = os.path.join(huc_path, filesname)
            gauge_id = str(filesname.split('_')[0]).zfill(8)
            
            df = pd.read_csv(files, sep=r'\s+', header=None)
            df = df[[1,2,3,4]]
            df.columns = ['year', 'month', 'day', 'flow_cfs']
            df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
            df['gauge_id'] = gauge_id
            
            #replace missing values with NaN and keep the gauge; basins with gaps inside the
            #study window are dropped later in make_model_dataset.py, which knows the dates
            df['flow_cfs'] = df['flow_cfs'].replace([-999, -999.0], np.nan)
            clean_df = df[['gauge_id', 'date', 'flow_cfs']]

            #append
            total_df.append(clean_df)
        
    return pd.concat(total_df, ignore_index = True)

In [6]:
#test1
test_streamflow_df = clean_usgs_streamflow(usgs_streamflow_dir, hucs=("17",))

print(test_streamflow_df.dtypes)

print(f'{test_streamflow_df.head()}')
test_streamflow_df

gauge_id               str
date        datetime64[us]
flow_cfs           float64
dtype: object
   gauge_id       date  flow_cfs
0  10396000 1980-01-01      54.0
1  10396000 1980-01-02      50.0
2  10396000 1980-01-03      50.0
3  10396000 1980-01-04      50.0
4  10396000 1980-01-05      75.0


,gauge_id,date,flow_cfs
0,10396000,1980-01-01,54.0
1,10396000,1980-01-02,50.0
2,10396000,1980-01-03,50.0
3,10396000,1980-01-04,50.0
4,10396000,1980-01-05,75.0
...,...,...,...
1135845,14400000,2014-12-27,3800.0
1135846,14400000,2014-12-28,3220.0
1135847,14400000,2014-12-29,2810.0
1135848,14400000,2014-12-30,2460.0


In [7]:
#test2
nan_rows = test_streamflow_df[test_streamflow_df['flow_cfs'].isna()]
print(nan_rows.head())

       gauge_id       date  flow_cfs
12760  10396000 2014-12-08       NaN
12761  10396000 2014-12-09       NaN
12762  10396000 2014-12-10       NaN
12763  10396000 2014-12-11       NaN
12764  10396000 2014-12-12       NaN


In [8]:
#join all three sources - one row per gauge per day
#input:  streamflow, daymet = DataFrames with gauge_id (text) and date
#        metadata = DataFrame with one row per gauge_id
#output: DataFrame, one row per gauge per day, with flow + weather + basin facts
#steps:
#1. merge streamflow and daymet on gauge_id + date (both are daily)
#2. merge metadata on gauge_id only, so basin facts repeat on every day
#   (many_to_one errors if a gauge appears twice in metadata)
def join_data(streamflow, daymet, metadata):
    daily = streamflow.merge(daymet, on=["gauge_id", "date"])
    joined = daily.merge(metadata, on="gauge_id", validate="many_to_one")
    
    #add runoff_mm_day to allow for basin normalization
    joined['runoff_mm_day'] = (joined['flow_cfs'] * 2.446575) / joined['drainage_area_km2']
    return joined

In [9]:
#save the final table
#input:  df = joined DataFrame from join_data, path = output file, e.g. "pnw_dataset.csv"
#output: writes the CSV (no row index), returns nothing
#note:   read it back with pd.read_csv(path, dtype={"gauge_id": str}) to keep leading zeros
def generate_csv(df, path):
    df.to_csv(path, index=False)

In [10]:
#run the whole pipeline - change hucs here to pick regions for every step
#until clean_daymet and clean_usgs_streamflow return DataFrames, this only shows metadata
hucs = ("17",)

metadata = clean_basin_metadata(basin_metadata_dir, hucs)
daymet = clean_daymet(daymet_mean_forcing_dir, hucs)
streamflow = clean_usgs_streamflow(usgs_streamflow_dir, hucs)

if daymet is None or streamflow is None:
    print("daymet or streamflow not finished yet - showing metadata only")
    display(metadata.head())
else:
    dataset = join_data(streamflow, daymet, metadata)
    generate_csv(dataset, "pnw_dataset.csv")
    print(dataset.shape)
    display(dataset.head())

daymet_data.shape=(1163344, 8)
(1135850, 22)


,gauge_id,date,flow_cfs,dayl(s),prcp(mm/day),srad(W/m2),tmax(C),tmin(C),vp(Pa),huc,...,lon,drainage_area_km2,area_km2,elevation_m,slope_m_per_km,frac_forest,mean_precip_mm_day,mean_pet_mm_day,mean_temp_c,runoff_mm_day
0,10396000,1980-01-01,54.0,32140.79,0.00,179.18,3.37,-6.81,380.85,17,...,-118.8675,528.89,969.57,1613.27,58.18941,0.0045,1.61,3.369,6.525,0.249797
1,10396000,1980-01-02,50.0,32140.79,0.00,232.28,6.55,-10.70,275.44,17,...,-118.8675,528.89,969.57,1613.27,58.18941,0.0045,1.61,3.369,6.525,0.231293
2,10396000,1980-01-03,50.0,32140.79,0.00,218.89,6.96,-8.13,341.11,17,...,-118.8675,528.89,969.57,1613.27,58.18941,0.0045,1.61,3.369,6.525,0.231293
3,10396000,1980-01-04,50.0,32276.78,0.00,207.61,5.64,-7.07,360.67,17,...,-118.8675,528.89,969.57,1613.27,58.18941,0.0045,1.61,3.369,6.525,0.231293
4,10396000,1980-01-05,75.0,32441.62,8.52,112.16,3.38,-4.20,457.17,17,...,-118.8675,528.89,969.57,1613.27,58.18941,0.0045,1.61,3.369,6.525,0.346940


In [11]:
#test3
print(f"Total rows in final dataset: {len(dataset):,}")
print(f"Total unique gauges remaining: {dataset['gauge_id'].nunique()}")
print("Remaining gauges:", dataset['gauge_id'].unique())

Total rows in final dataset: 1,135,850
Total unique gauges remaining: 91
Remaining gauges: <StringArray>
['10396000', '12010000', '12013500', '12020000', '12025000', '12025700',
 '12035000', '12040500', '12041200', '12043000', '12048000', '12054000',
 '12056500', '12073500', '12082500', '12092000', '12095000', '12114500',
 '12115000', '12115500', '12117000', '12141300', '12143600', '12144000',
 '12145500', '12147500', '12147600', '12167000', '12175500', '12178100',
 '12186000', '12189500', '12358500', '12374250', '12375900', '12377150',
 '12381400', '12383500', '12388400', '12390700', '12411000', '12414500',
 '12447390', '12451000', '12488500', '13011500', '13011900', '13018300',
 '13023000', '13083000', '13161500', '13235000', '13240000', '13310700',
 '13313000', '13331500', '13337000', '13338500', '13340000', '13340600',
 '14020000', '14092750', '14096850', '14137000', '14138800', '14138870',
 '14138900', '14139800', '14141500', '14154500', '14158500', '14158790',
 '14166500', '14182